# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    last = None
    for attempt in range(tries): # TODO: loop up to `tries` times
        try:
            r = requests.get(BASE, params=params, timeout=15) # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200

            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            if r.status_code not in RETRY_STATUSES: # TODO:   raise on a status that is not in RETRY_STATUSES

                raise RuntimeError(f'not retryable: {r.status_code} {r.text[:200]}')
            last = f'status {r.status_code}'
        except requests.RequestException as e:
            last = repr(e)
        if attempt < tries - 1: # TODO:   otherwise sleep (increasing each time) and try again
            delay = 0.5 * (2 ** attempt)
            print(f'attempt {attempt + 1} failed ({last}); retrying in {delay}s')
            time.sleep(delay)
    raise RuntimeError(f'gave up after {tries} attempts: {last}') # TODO: raise if every attempt failed

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
weekends = ['2026-09-05', '2026-10-03']

all_wx = get_weather('2026-09-05', '2026-10-03')
all_wx['time'] = pd.to_datetime(all_wx['time'])

wx = all_wx[all_wx['time'].dt.strftime('%Y-%m-%d').isin(weekends)].copy()
wx = wx.reset_index(drop=True)

used_fallback = False

print('rows in full range:', len(all_wx))
print('rows kept:', len(wx))
print('used_fallback:', used_fallback)
wx

rows in full range: 29
rows kept: 2
used_fallback: False


,time,temperature_2m_max,precipitation_sum
0,2026-09-05,29.4,2.5
1,2026-10-03,22.0,14.7


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'

weekends = ['2026-09-05', '2026-10-03']

used_fallback = False
try:
    all_wx = get_weather('2026-09-05', '2026-10-03')
    all_wx['time'] = pd.to_datetime(all_wx['time'])
    wx = all_wx[all_wx['time'].dt.strftime('%Y-%m-%d').isin(weekends)].copy()
    wx = wx.reset_index(drop=True)
    wx['source'] = 'open-meteo'
except Exception as e:
    # the call genuinely failed: say so out loud, and record it
    used_fallback = True
    print(f'FALLING BACK to sample data: {e}')
    wx = FALLBACK.copy()
    wx['time'] = pd.to_datetime(wx['time'])
    wx['source'] = 'fallback sample'

print('used_fallback:', used_fallback)
wx

used_fallback: False


,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
# TODO
wx = (wx
    .rename(columns={
        'time': 'date',
        'temperature_2m_max': 'temp_max',
        'precipitation_sum': 'precip_mm',
    })
    [['date', 'temp_max', 'precip_mm', 'source']]
    .copy())

wx['date'] = pd.to_datetime(wx['date'])

wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
# TODO
wetter = wx.loc[wx['precip_mm'].idxmax()]
drier = wx.loc[wx['precip_mm'].idxmin()]
diff = wetter['precip_mm'] - drier['precip_mm']

print(f"{wetter['date']:%b %d} was the wetter home weekend with {wetter['precip_mm']:.1f}mm of rain, "
      f"vs {drier['precip_mm']:.1f}mm on {drier['date']:%b %d}, "
      f"a difference of {diff:.1f}mm.")

Oct 03 was the wetter home weekend with 14.7mm of rain, vs 2.5mm on Sep 05, a difference of 12.2mm.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
# TODO
from datetime import datetime

print('Source:   Open-Meteo historical archive API')
print('Endpoint:', BASE)
print('Coords:  ', 38.03, -78.51, '(Charlottesville)')
print('Range:   ', '2026-09-05 to 2026-10-03 (kept 2026-09-05 and 2026-10-03)')
print('Pulled:  ', datetime.now().strftime('%Y-%m-%d %H:%M'))
print('Data:    ', 'FALLBACK SAMPLE (not real)' if used_fallback else 'real (live API)')

Source:   Open-Meteo historical archive API
Endpoint: https://archive-api.open-meteo.com/v1/archive
Coords:   38.03 -78.51 (Charlottesville)
Range:    2026-09-05 to 2026-10-03 (kept 2026-09-05 and 2026-10-03)
Pulled:   2026-10-12 01:40
Data:     real (live API)


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty dataframe on failure, the merge would not find any weather rows to match against the 200 days of sales. Rain would come back as NaN for every day, so the chart would show no rain at all and it would look like rain has nothing to do with revenue. Nobody would catch it because nothing crashes and no error prints, and an empty frame from a failed request looks the same as an empty frame from a dry stretch. The report would say “rain doesn’t affect sales” when the real problem was actually a failed api call.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# stretch
def build_city(city, lat, lon):
    raw = get_weather('2026-09-05', '2026-10-03', lat=lat, lon=lon)
    raw['time'] = pd.to_datetime(raw['time'])
    keep = raw[raw['time'].dt.strftime('%Y-%m-%d').isin(weekends)].copy()
    keep = (keep.rename(columns={'time': 'date',
                                 'temperature_2m_max': 'temp_max',
                                 'precipitation_sum': 'precip_mm'})
                [['date', 'temp_max', 'precip_mm']])
    keep['city'] = city
    keep['source'] = 'open-meteo'
    return keep

cville = build_city('Charlottesville', 38.03, -78.51)
orlando = build_city('Orlando', 28.54, -81.38)

both = pd.concat([cville, orlando], ignore_index=True)
both = both[['city', 'date', 'temp_max', 'precip_mm', 'source']]

assert len(both) == 4
assert set(both['city']) == {'Charlottesville', 'Orlando'}
both

,city,date,temp_max,precip_mm,source
0,Charlottesville,2026-09-05,29.4,2.5,open-meteo
1,Charlottesville,2026-10-03,22.0,14.7,open-meteo
2,Orlando,2026-09-05,32.1,3.9,open-meteo
3,Orlando,2026-10-03,30.4,14.5,open-meteo
